In [1]:
import re
import spacy
from tqdm import tqdm

CITATION_PATTERN = re.compile(
    r"""
    (
        \[\s*\d+(?:\s*[-–]\s*\d+)?(?:\s*,\s*\d+)*\s*\]      # [12], [3,7], [4-8]
      |
        \((?:[A-Z][A-Za-z]+(?:\s+et al\.)?(?:\s*&\s*[A-Z][A-Za-z]+)?,\s*\d{4}
            (?:;\s*[A-Z][A-Za-z]+(?:\s+et al\.)?(?:\s*&\s*[A-Z][A-Za-z]+)?,\s*\d{4})*
        )\)                                                 # (Smith, 2020; Brown, 2019)
      |
        \([A-Z][A-Za-z]+\s+and\s+[A-Z][A-Za-z]+,\s*\d{4}\)  # (Smith and Jones, 2019)
    )
    """,
    re.VERBOSE,
)

nlp = spacy.load("en_core_web_sm")

def split_sentences_with_offsets(text):
    doc = nlp(text)
    out = []
    for sent in doc.sents:
        sent_text = text[sent.start_char:sent.end_char]
        if sent_text.strip() == "":
            continue
        out.append({"text": sent_text, "start": sent.start_char, "end": sent.end_char})
    return out

def has_citation(sentence):
    return CITATION_PATTERN.search(sentence) is not None

def overlaps(a_start, a_end, b_start, b_end):
    return not (a_end <= b_start or b_end <= a_start)

def clip_span_to_sentence(span_start, span_end, sent_start, sent_end):
    return max(span_start, sent_start), min(span_end, sent_end)

def build_format_sentence_dataset(format_items, category_name = "Format") :
    out_rows = []

    for doc_idx, item in enumerate(tqdm(format_items, desc=f"Processing {category_name}")):
        doc_text = item.get("document", "") or ""
        issue_spans = item.get("spans", []) or []

        sentences = split_sentences_with_offsets(doc_text)

        for sent_idx, sent in enumerate(sentences):
            sent_text = sent["text"]
            sent_start = sent["start"]
            sent_end = sent["end"]

            overlapping_issue_spans = []
            for sp in issue_spans:
                sp_start = int(sp["start"])
                sp_end = int(sp["end"])

                if overlaps(sp_start, sp_end, sent_start, sent_end):
                    clipped_start, clipped_end = clip_span_to_sentence(sp_start, sp_end, sent_start, sent_end)
                    local_start = clipped_start - sent_start
                    local_end = clipped_end - sent_start

                    overlapping_issue_spans.append({
                        "doc_start": sp_start,
                        "doc_end": sp_end,
                        "sent_start": local_start,
                        "sent_end": local_end,
                        "text": sent_text[local_start:local_end],
                        "span_text_from_source": sp.get("span", ""),
                    })

            if overlapping_issue_spans:
                label = 1 
            else:
                if not has_citation(sent_text):
                    continue
                label = 0

            out_rows.append({
                "category": category_name,
                "doc_index": doc_idx,
                "sent_index": sent_idx,
                "label": label,
                "sentence": sent_text,
                "sentence_char_start_in_doc": sent_start,
                "sentence_char_end_in_doc": sent_end,
                "issue_spans_in_sentence": overlapping_issue_spans,
            })

    return out_rows


In [2]:
import json

synth_path = "../../../synthetic_sampling/samples/synthetic_samples_complete.json"
real_path = "../../../synthetic_sampling/samples/total_real_data.json"
synth_out_path = "../../data/format/format_sentence_synth.json"
real_out_path = "../../data/format/format_sentence_real.json"

with open(synth_path, "r", encoding="utf-8") as f:
    span_data_all = json.load(f)

with open(real_path, "r", encoding="utf-8") as f:
    real_data_all = json.load(f)

synth_format = span_data_all["Format"]
real_format = real_data_all['Format']

synth_rows = build_format_sentence_dataset(synth_format, category_name="Format")
real_rows = build_format_sentence_dataset(real_format, category_name="Format")

print(f"SYNTH Total sentence samples kept (citation-only): {len(synth_rows)}")
print(f"Positives: {sum(r['label'] for r in synth_rows)} | Negatives: {sum(1-r['label'] for r in synth_rows)}")

print(f"REAL Total sentence samples kept (citation-only): {len(synth_rows)}")
print(f"Positives: {sum(r['label'] for r in real_rows)} | Negatives: {sum(1-r['label'] for r in real_rows)}")


with open(synth_out_path, "w", encoding="utf-8") as f:
    json.dump(synth_rows, f, indent=2, ensure_ascii=False)

with open(real_out_path, "w", encoding="utf-8") as f:
    json.dump(real_rows, f, indent=2, ensure_ascii=False)

print(f"Wrote: {synth_out_path} and {real_out_path}")

Processing Format: 100%|██████████| 22/22 [00:02<00:00,  8.84it/s]

SYNTH Total sentence samples kept (citation-only): 690
Positives: 402 | Negatives: 288
REAL Total sentence samples kept (citation-only): 690
Positives: 32 | Negatives: 27
Wrote: ../../data/format/format_sentence_synth.json and ../../data/format/format_sentence_real.json


In [3]:
def clean_format_sentence_record(rec):
    sentence = rec.get("sentence", "")
    spans = rec.get("issue_spans_in_sentence", []) or []

    if not spans:
        return None  # DROP negatives

    cleaned_spans = []

    for sp in spans:
        start = int(sp.get("sent_start", 0))
        end = int(sp.get("sent_end", 0))

        start = max(0, min(start, len(sentence)))
        end = max(0, min(end, len(sentence)))

        if end < start:
            start, end = end, start

        cleaned_spans.append({
            "start": start,
            "end": end,
            "text": sentence[start:end],
        })

    return {
        "sentence": sentence,
        "format_spans": cleaned_spans,
    }

def clean_format_dataset(records):
    cleaned = []
    for r in records:
        c = clean_format_sentence_record(r)
        if c is not None:
            cleaned.append(c)
    return cleaned


In [4]:
train_data = clean_format_dataset(synth_rows)
eval_data = clean_format_dataset(real_rows)

In [5]:
import re
from collections import defaultdict
from typing import List, Tuple, Dict, Any, Optional

def tokenize_with_offsets(text: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    tokens = []
    offsets = []
    for match in re.finditer(r"\S+", text):
        tokens.append(match.group())
        offsets.append((match.start(), match.end()))
    return tokens, offsets

def _token_overlaps(tok_start: int, tok_end: int, span_start: int, span_end: int) -> bool:
    return not (tok_end <= span_start or tok_start >= span_end)

def _find_span_occurrence(document_text: str, span_text: str, used_starts: set) -> Tuple[int, int]:
    if not span_text:
        return -1, -1

    start = document_text.find(span_text)
    while start != -1:
        if start not in used_starts:
            end = start + len(span_text)
            return start, end
        start = document_text.find(span_text, start + 1)

    return -1, -1

def _normalize_spans_for_document(item: Dict[str, Any], document_text: str) -> List[Tuple[int, int]]:
    normalized: List[Tuple[int, int]] = []
    used_starts = set()

    # --------
    # Format C: sentence object with "format_spans"
    # --------
    if "format_spans" in item and isinstance(item["format_spans"], list):
        for s in item["format_spans"]:
            s_start: Optional[int] = None
            s_end: Optional[int] = None

            if isinstance(s, dict):
                # prefer explicit start/end
                if "start" in s and "end" in s:
                    try:
                        s_start = int(s["start"])
                        s_end = int(s["end"])
                    except Exception:
                        s_start, s_end = None, None

                # if invalid, fall back to finding provided text
                if s_start is None or s_end is None or s_start >= s_end:
                    span_text = s.get("text") or s.get("span") or s.get("span_text")
                    if isinstance(span_text, str) and span_text.strip():
                        s_start, s_end = _find_span_occurrence(document_text, span_text, used_starts)

            elif isinstance(s, str) and s.strip():
                s_start, s_end = _find_span_occurrence(document_text, s, used_starts)

            if s_start is None or s_end is None:
                continue

            # clamp
            s_start = max(0, min(len(document_text), s_start))
            s_end = max(0, min(len(document_text), s_end))
            if s_start >= s_end:
                continue

            used_starts.add(s_start)
            normalized.append((s_start, s_end))

        return normalized

    # --------
    # Format A: per-document object with "spans"
    # --------
    if "spans" in item and isinstance(item["spans"], list):
        for s in item["spans"]:
            s_start: Optional[int] = None
            s_end: Optional[int] = None

            if isinstance(s, dict):
                if "start" in s and "end" in s:
                    try:
                        s_start = int(s["start"])
                        s_end = int(s["end"])
                    except Exception:
                        s_start, s_end = None, None

                if s_start is None or s_end is None or s_start >= s_end:
                    span_text = s.get("span") or s.get("text") or s.get("span_text")
                    if isinstance(span_text, str) and span_text.strip():
                        s_start, s_end = _find_span_occurrence(document_text, span_text, used_starts)

            elif isinstance(s, str) and s.strip():
                s_start, s_end = _find_span_occurrence(document_text, s, used_starts)

            if s_start is None or s_end is None:
                continue

            s_start = max(0, min(len(document_text), s_start))
            s_end = max(0, min(len(document_text), s_end))
            if s_start >= s_end:
                continue

            used_starts.add(s_start)
            normalized.append((s_start, s_end))

        return normalized

    # --------
    # Format B: per-span object (legacy)
    # --------
    try:
        s_start = int(item.get("start"))
        s_end = int(item.get("end"))
    except Exception:
        s_start, s_end = None, None

    if s_start is None or s_end is None or s_start >= s_end:
        span_text = item.get("span") or item.get("text") or item.get("span_text")
        if isinstance(span_text, str) and span_text.strip():
            s_start, s_end = _find_span_occurrence(document_text, span_text, used_starts)

    if s_start is None or s_end is None:
        return []

    s_start = max(0, min(len(document_text), s_start))
    s_end = max(0, min(len(document_text), s_end))
    if s_start >= s_end:
        return []

    return [(s_start, s_end)]

def convert_to_ner_format(span_data: List[dict], label: str) -> List[dict]:
    docs = defaultdict(list)

    # Group items by text.
    for item in span_data:
        if not isinstance(item, dict):
            continue

        # Prefer document/full_text/text; fall back to sentence for your new format
        doc_text = (
            item.get("document")
            or item.get("full_text")
            or item.get("text")
            or item.get("sentence")
        )

        if not isinstance(doc_text, str) or not doc_text.strip():
            continue

        docs[doc_text].append(item)

    ner_examples = []
    label_name = label.replace(" ", "_")

    for document_text, items_for_doc in docs.items():
        tokens, offsets = tokenize_with_offsets(document_text)
        labels = ["O"] * len(tokens)

        # Collect all spans for this doc/sentence
        all_spans: List[Tuple[int, int]] = []
        for it in items_for_doc:
            all_spans.extend(_normalize_spans_for_document(it, document_text))

        all_spans.sort(key=lambda x: (x[0], x[1]))

        # Apply BIO labels
        for span_start, span_end in all_spans:
            inside = False
            for i, (tok_start, tok_end) in enumerate(offsets):
                if not _token_overlaps(tok_start, tok_end, span_start, span_end):
                    if inside:
                        inside = False
                    continue

                if labels[i] == "O":
                    if not inside:
                        labels[i] = f"B-{label_name}"
                        inside = True
                    else:
                        labels[i] = f"I-{label_name}"
                else:
                    # don't overwrite existing labels, but maintain inside state
                    if not inside:
                        inside = True

        ner_examples.append({"tokens": tokens, "labels": labels})

    return ner_examples


In [6]:
format_train_path = "../../data/format/format_ner_train.json"
format_eval_path = "../../data/format/format_ner_eval.json"

train_ner = convert_to_ner_format(train_data, label="FORMAT")
eval_ner = convert_to_ner_format(eval_data, label="FORMAT")


with open(format_train_path, 'w') as f:
    json.dump(train_ner, f, indent=2)
    
with open(format_eval_path, 'w') as f:
    json.dump(eval_ner, f, indent=2)

print(f"Wrote: {format_train_path} and {format_eval_path}")

Wrote: ../../data/format/format_ner_train.json and ../../data/format/format_ner_eval.json
